[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Boutique_Fisica_Avanzada/Trazado_Rayos_Agujero_Negro.ipynb)



# Relatividad General: Lente Gravitacional de un Agujero Negro
Cuando la luz pasa cerca de un campo gravitacional extremo (como el Sol o un Agujero Negro), la métrica del espacio-tiempo se curva y los fotones ya no viajan en "líneas rectas euclidianas", sino que siguen **Geodésicas Nulas** curvas.

Resolveremos las ecuaciones de movimiento geodésico para la métrica de **Schwarzschild** y trazaremos las trayectorias fotónicas masivas. Mostrando cómo un agujero negro deforma visualmente el espacio a su alrededor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Ecuaciones Geodésicas en el plano ecuatorial (theta = pi/2)
# Usando la variable u = 1/r, la ecuación de la órbita del fotón se simplifica a la ecuación de Binet:
# d^2u / dphi^2 + u = 3/2 * r_s * u^2
# Donde r_s es el radio de Schwarzschild (r_s = 2GM/c^2). Normalizamos r_s = 1.

def binet_geodesic(y, phi):
    u, du_dphi = y
    # y[0] = u, y[1] = du/dphi
    d2u_dphi2 = 1.5 * u**2 - u
    return [du_dphi, d2u_dphi2]

# Rayos impactando desde el infinito (x = -inf) viajando paralelos hacia la derecha (+x)
# Su parámetro de impacto (distancia perpendicular al agujero negro en y) es 'b'
impact_parameters = np.linspace(1.0, 6.0, 25)

phi_eval = np.linspace(np.pi, -2*np.pi, 2000)  # Desde infinito izquierdo integrando la curva

plt.style.use('dark_background')
plt.figure(figsize=(8, 8))

# Círculo del Horizonte de Sucesos (r = 1) y Esfera de Fotones (r = 1.5)
circle_rs = plt.Circle((0, 0), 1.0, color='black', zorder=10)
circle_ph = plt.Circle((0, 0), 1.5, color='orange', alpha=0.3, ls='--', zorder=9)
plt.gca().add_patch(circle_rs)
plt.gca().add_patch(circle_ph)

for b in impact_parameters:
    # Condiciones iniciales en el infinito (phi = pi)
    # u(pi) = 0 (r=inf), du/dphi = 1/b (proveniente de las relaciones geométricas asintóticas)
    y0 = [0.0, 1.0/b]
    sol = odeint(binet_geodesic, y0, phi_eval)
    u = sol[:, 0]
    
    # Filtrar solo valores físicos (r > 0 -> u > 0)
    valid = u > 0
    u_valid = u[valid]
    phi_valid = phi_eval[valid]
    r = 1.0 / u_valid
    
    # Si r cae por debajo de r_s (1.0), el fotón es tragado
    mask_outside = r > 1.0
    r_plot = r[mask_outside]
    phi_plot = phi_valid[mask_outside]
    
    # Convertir a cartesianas
    x = r_plot * np.cos(phi_plot)
    y = r_plot * np.sin(phi_plot)
    
    plt.plot(x, y, color='cyan', alpha=0.7, lw=1)

# Anotaciones
plt.xlim(-10, 10)
plt.ylim(-10, 10)
plt.gca().set_aspect('equal')
plt.title("Geodésicas Fotónicas en Relatividad General (Métrica de Schwarzschild)")
plt.plot([], [], color='black', label='Horizonte Eventos ($1 r_s$)')
plt.plot([], [], color='orange', alpha=0.5, label='Esfera Fotónica ($1.5 r_s$)')
plt.legend(loc='lower left')
plt.show()

print("Rayos lejanos se desvían levemente (Lente Débil).")
print("Rayos a 2.6 r_s (parámetro de impacto crítico 3*sqrt(3)/2) orbitan eternamente en la esfera fotónica.")
print("Rayos con impacto menor son devorados sin retorno.")